# 01 — Installation du projet PIR
## Windows + macOS + Linux — Conda — Python 3.15.2

Ce notebook est le guide d'installation commun du projet.

Objectif :

```text
même environnement
mêmes dépendances
même version de Python
même kernel VS Code / Jupyter
```

La version retenue est :

```text
Python 3.15.2
```

Elle est disponible via `conda-forge` sur Windows, macOS Intel, macOS Apple Silicon
et Linux.

Le gestionnaire d'environnement utilisé dans ce projet est désormais :

```text
conda
```

> Distribution recommandée : **Miniforge** ou **Miniconda**.
> Les commandes du projet restent des commandes `conda`.

# 0 — Compatibilité Python 3.15.2

État vérifié pour la stack actuelle du projet :

| Paquet | Python 3.15 |
|---|---|
| numpy | Oui |
| pandas | Oui |
| scipy | Oui |
| shapely | Oui |
| pyproj | Oui |
| matplotlib | Oui |
| pydantic | Oui |
| requests | Oui |
| pytest | Oui |
| ipykernel | Compatible |
| gpxpy | Pur Python / `py3-none-any`, Python >= 3.6 |

Les bibliothèques compilées critiques disposent de builds CPython 3.15.

`gpxpy` est plus ancien et ne publie pas de classifier spécifique 3.15, mais son
wheel est universel Python 3 et il ne contient pas d'extension native. Une cellule
de test est fournie plus bas.

Pour le projet actuel, on fixe donc :

```text
python=3.15.2
```

# 1 — Pré-requis généraux

Installer sur la machine :

1. **Git**
2. **Visual Studio Code**
3. Extensions VS Code :
   - Python
   - Jupyter
4. **Conda**
5. **Ollama**

Le projet fonctionne sur :

```text
Windows 10/11
macOS Intel
macOS Apple Silicon
Linux x86_64
Linux ARM64 si les paquets de la plateforme sont disponibles
```

# 2 — Installer Conda

## Option recommandée : Miniforge

Miniforge fournit `conda` et utilise naturellement l'écosystème `conda-forge`.

Site officiel :

```text
https://github.com/conda-forge/miniforge
```

## Alternative : Miniconda

Documentation officielle Conda :

```text
https://docs.conda.io/
```

Après installation, **fermer puis rouvrir le terminal**.

### Windows

Utiliser de préférence :

```text
Anaconda Prompt
ou
PowerShell après `conda init powershell`
```

### macOS

Terminal par défaut : `zsh`.

Si nécessaire :

```bash
conda init zsh
```

puis fermer/réouvrir le Terminal.

### Linux

Pour bash :

```bash
conda init bash
```

puis fermer/réouvrir le terminal.

# 3 — Vérifier Conda

Dans un terminal :

```bash
conda --version
conda info
```

Une version de Conda doit apparaître.

Si `conda: command not found` apparaît sur macOS/Linux, le shell n'a probablement
pas encore été initialisé.

Si PowerShell ne reconnaît pas `conda` sous Windows, ouvrir **Anaconda Prompt**
ou exécuter l'initialisation Conda depuis le terminal fourni par l'installateur.

# 4 — Créer l'environnement PIR

Se placer à la racine du projet, là où se trouve :

```text
environment.yml
```

Puis exécuter :

```bash
conda env create -f environment.yml
```

Activer ensuite :

```bash
conda activate pir-route
```

Vérifier :

```bash
python --version
```

Résultat attendu :

```text
Python 3.15.2
```

## Si l'environnement existe déjà

Mettre les dépendances à jour :

```bash
conda env update -n pir-route -f environment.yml --prune
```

# 5 — Création manuelle sans `environment.yml`

Si vous ne voulez pas utiliser le fichier YAML :

```bash
conda create -n pir-route -c conda-forge --override-channels   python=3.15.2   numpy pandas scipy   gpxpy shapely pyproj   pydantic requests matplotlib pytest   ipykernel   -y
```

Sous **PowerShell**, si le `\` multi-ligne pose problème, mettre toute la commande
sur une seule ligne :

```powershell
conda create -n pir-route -c conda-forge --override-channels python=3.15.2 numpy pandas scipy gpxpy shapely pyproj pydantic requests matplotlib pytest ipykernel -y
```

Puis :

```bash
conda activate pir-route
```

# 6 — Enregistrer le kernel Jupyter

Toujours avec :

```text
(pir-route)
```

visible dans le terminal :

```bash
python -m ipykernel install --user --name pir-route --display-name "Python 3.15.2 — PIR"
```

Cela permet à VS Code de sélectionner explicitement le bon environnement.

# 7 — Sélectionner le bon kernel dans VS Code

Dans VS Code :

1. ouvrir le dossier du projet ;
2. ouvrir un notebook `.ipynb` ;
3. cliquer en haut à droite sur **Select Kernel** ;
4. choisir :

```text
Python 3.15.2 — PIR
```

ou l'interpréteur Conda :

```text
pir-route
```

## Très important

Si vous obtenez :

```text
ModuleNotFoundError: No module named 'numpy'
```

alors le problème est généralement **le kernel sélectionné**, pas NumPy.

Vérifier dans une cellule :

```python
import sys
print(sys.executable)
print(sys.version)
```

Le chemin doit pointer vers l'environnement `pir-route`.

# 8 — Test complet de l'environnement Python

In [ ]:
import sys
import platform
import importlib

print("Python :", sys.version)
print("Executable :", sys.executable)
print("OS :", platform.platform())
print()

packages = [
    "numpy",
    "pandas",
    "scipy",
    "gpxpy",
    "shapely",
    "pyproj",
    "pydantic",
    "requests",
    "matplotlib",
    "pytest",
    "ipykernel",
]

failed = []

for name in packages:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", "version non exposée")
        print(f"OK  {name:12s} {version}")
    except Exception as exc:
        failed.append((name, repr(exc)))
        print(f"KO  {name:12s} {exc}")

print()

if sys.version_info[:3] != (3, 15, 2):
    print(
        "ATTENTION : le kernel actif n'utilise pas exactement Python 3.15.2."
    )
else:
    print("OK — Python 3.15.2 actif.")

if failed:
    raise RuntimeError(
        "Certaines dépendances ne fonctionnent pas : "
        + repr(failed)
    )

print("OK — toutes les dépendances Python principales sont importables.")

# 9 — Petit test géospatial

Ce test vérifie les bibliothèques les plus importantes pour les notebooks GPX / OSM.

In [ ]:
from shapely.geometry import LineString, Polygon
from pyproj import Transformer

route_test = LineString([
    (1.0, 43.0),
    (1.001, 43.001),
])

forest_test = Polygon([
    (0.999, 42.999),
    (1.002, 42.999),
    (1.002, 43.002),
    (0.999, 43.002),
])

assert route_test.intersects(forest_test)

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True,
)

x, y = transformer.transform(1.0, 43.0)

print("Shapely intersection : OK")
print("PyProj transformation : OK")
print("Coordonnée projetée test :", round(x, 2), round(y, 2))

# 10 — Installer Ollama

Ollama est indépendant de l'environnement Conda.

## Windows — PowerShell

```powershell
irm https://ollama.com/install.ps1 | iex
```

On peut aussi utiliser l'installateur graphique officiel.

## macOS

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

ou installer l'application Ollama officielle.

## Linux

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

Après installation :

```bash
ollama --version
```

# 11 — Télécharger le modèle utilisé par le projet

Le modèle actuel du pipeline est :

```text
llama3.1:8b
```

Dans un terminal :

```bash
ollama pull llama3.1:8b
```

Puis :

```bash
ollama list
```

Vous devez voir `llama3.1:8b`.

Test rapide :

```bash
ollama run llama3.1:8b "Réponds uniquement : PIR OK"
```

# 12 — Tester l'API locale Ollama

L'API utilisée par les notebooks est :

```text
http://localhost:11434/api/chat
```

In [ ]:
import requests

url = "http://localhost:11434/api/chat"

payload = {
    "model": "llama3.1:8b",
    "messages": [
        {
            "role": "user",
            "content": "Réponds uniquement avec : OLLAMA_OK",
        }
    ],
    "stream": False,
}

try:
    response = requests.post(
        url,
        json=payload,
        timeout=120,
    )

    print("HTTP :", response.status_code)

    if response.status_code == 200:
        data = response.json()
        print("Réponse :", data["message"]["content"])
    else:
        print(response.text)

except requests.RequestException as exc:
    print("Ollama inaccessible :", exc)
    print(
        "Vérifier qu'Ollama est installé, lancé, "
        "et que llama3.1:8b est téléchargé."
    )

# 13 — Tester l'accès OSM / Overpass

Le notebook 03bis utilise des serveurs publics Overpass.

Une erreur `502`, `503`, `504` ou `429` peut être **temporaire** et ne signifie pas
que l'installation locale est cassée.

On teste plusieurs instances.

In [ ]:
import requests

OVERPASS_TEST_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

test_query = '''
[out:json][timeout:20];
node(0,0,0.001,0.001);
out count;
'''

working_endpoint = None

for endpoint in OVERPASS_TEST_ENDPOINTS:
    try:
        r = requests.post(
            endpoint,
            data={"data": test_query},
            headers={
                "User-Agent":
                "PIR-Route-Agent/installation-test"
            },
            timeout=30,
        )

        print(endpoint, "->", r.status_code)

        if r.status_code == 200:
            working_endpoint = endpoint
            break

    except requests.RequestException as exc:
        print(endpoint, "-> erreur :", exc)

if working_endpoint:
    print("OVERPASS OK :", working_endpoint)
else:
    print(
        "Aucune instance Overpass ne répond actuellement. "
        "Cela peut être temporaire."
    )

# 14 — Structure minimale du projet

La racine doit contenir approximativement :

```text
PIR/
├── 01_INSTALLATION...
├── 02_...
├── 03_...
├── 03BIS_...
├── 04_...
├── 05_...
├── environment.yml
└── data/
    ├── input/
    ├── output/
    └── cache/
```

Créer les dossiers manquants si nécessaire.

In [ ]:
from pathlib import Path

for folder in [
    Path("data/input"),
    Path("data/output"),
    Path("data/cache"),
]:
    folder.mkdir(parents=True, exist_ok=True)
    print("OK :", folder)

# 15 — Erreurs fréquentes

## `ModuleNotFoundError`

Cause la plus probable :

```text
mauvais kernel VS Code
```

Solution :

```text
Select Kernel
-> Python 3.15.2 — PIR
```

---

## `conda: command not found`

Fermer/réouvrir le terminal après installation.

Sinon :

```bash
conda init zsh
```

sur macOS, ou :

```bash
conda init bash
```

sur Linux.

Sous Windows, utiliser Anaconda Prompt ou initialiser PowerShell.

---

## Ollama retourne 404

Vérifier :

```bash
ollama list
```

et que le nom du modèle correspond exactement à :

```text
llama3.1:8b
```

---

## Overpass retourne 502 / 504

Ce n'est pas nécessairement un bug.

Les notebooks récents testent plusieurs serveurs et utilisent un cache local.

---

## Le collègue et moi n'avons pas les mêmes versions

Ne pas installer les dépendances globalement.

Utiliser :

```bash
conda env update -n pir-route -f environment.yml --prune
```

# 16 — Reproductibilité

Pour voir l'environnement :

```bash
conda list
```

Pour exporter uniquement les dépendances explicitement demandées :

```bash
conda env export -n pir-route --from-history > environment.from-history.yml
```

Pour reconstruire l'environnement du projet, la référence reste le fichier
`environment.yml` fourni avec le dépôt.

# 17 — Validation finale

Avant de lancer le notebook 02, vérifier :

```text
[ ] conda fonctionne
[ ] environnement pir-route activable
[ ] Python = 3.15.2
[ ] kernel VS Code = Python 3.15.2 — PIR
[ ] imports Python OK
[ ] test Shapely / PyProj OK
[ ] Ollama installé
[ ] llama3.1:8b téléchargé
[ ] API Ollama accessible
[ ] au moins un serveur Overpass accessible
[ ] dossiers data/input, data/output, data/cache présents
```

Une fois ces points validés, le poste est prêt pour le pipeline PIR.